# SEM particle annotation


In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
nb_path = '/content/notebooks'
models_path = '/content/checkpoints'
!mkdir -p $models_path
!mkdir -p $nb_path

#os.symlink('/content/drive/My Drive/Colab Notebooks/models', models_path)
#os.symlink('/content/drive/My Drive/Colab Notebooks/packages', nb_path)
sys.path.insert(0,nb_path)

Mounted at /content/drive


In [ ]:
using_colab = True


In [ ]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    #!{sys.executable} -m pip install opencv-python matplotlib
    #!{sys.executable} -m pip install --target=$nb_path 'git+https://github.com/facebookresearch/sam2.git'
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam2.git'

    !mkdir -p $models_path
    !wget -nc -P $models_path https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt


PyTorch version: 2.6.0+cu124
Torchvision version: 0.21.0+cu124
CUDA is available: True
  Cloning https://github.com/facebookresearch/sam2.git to /tmp/pip-req-build-l3ajoeir
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/sam2.git /tmp/pip-req-build-l3ajoeir
  Resolved https://github.com/facebookresearch/sam2.git to commit 2b90b9f5ceec907a1c18123530e92e794ad901a4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

if not os.path.isdir('./NanoparticlesSAM'):
    !git clone https://github.com/Biolographer/NanoparticlesSAM.git

module_path = os.path.abspath(os.path.join('NanoparticlesSAM/NanoparticlesSAM'))

#module_path = os.path.abspath(os.path.join('./NanoparticlesSAM'))
sys.path.append(module_path)

Cloning into 'NanoparticlesSAM'...
remote: Enumerating objects: 369, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 369 (delta 108), reused 108 (delta 48), pack-reused 182 (from 2)
Receiving objects: 100% (369/369), 134.94 MiB | 19.02 MiB/s, done.
Resolving deltas: 100% (174/174), done.


In [ ]:
from NanoparticlesSAM import *
from particle_seg import *
from particle_loader import *
from plots import plot_seg_mask, plot_rect
from dataset import *

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import glob
from tqdm import tqdm

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from PIL import Image

#%matplotlib tk
import matplotlib.image as mpimg

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())


PyTorch version: 2.6.0+cu124
Torchvision version: 0.21.0+cu124
CUDA is available: True


# configure model

In [ ]:
# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

using device: cuda


In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_checkpoint = "/content/checkpoints/sam2.1_hiera_tiny.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_t.yaml"
#model_cfg = '/content/notebooks/sam2/configs/sam2.1/sam2.1_hiera_l.yaml'
#model_cfg = '/content/sam2.1_hiera_l.yaml'

sam2 = build_sam2(model_cfg, sam2_checkpoint,
                  device=device, apply_postprocessing=False)

predictor = SAM2ImagePredictor(sam2)

# Set up dataset



In [ ]:
TRAIN_IMAGE_FOLDER = '/content/drive/MyDrive/shield_data/training data'
TEST_IMAGE_FOLDER = '/content/drive/MyDrive/shield_data/testing data'

In [ ]:
data = CircleMaskDataset(TRAIN_IMAGE_FOLDER, metadata_fn=get_circle_metadata, error_mode='silent')

In [ ]:
particles = 0
for image, mask in data:
    particles += int(mask.max())
print(particles)

1772


In [ ]:
LOG_FILE = f'{models_path}/training_log.csv'

!python /content/NanoparticlesSAM/NanoparticlesSAM/train.py \
  --TRAIN_IMAGE_FOLDER "{TRAIN_IMAGE_FOLDER}" \
  --TEST_IMAGE_FOLDER "{TEST_IMAGE_FOLDER}" \
  --sam2_checkpoint "{sam2_checkpoint}" \
  --CHECKPOINT_DIR "{models_path}" \
  --LOG_FILE "{LOG_FILE}"

In [ ]:
import matplotlib.pyplot as plt

x = data.__getitem__(140)
fig, axes = plt.subplots(1, 2, figsize=(10, 5)) # Create a figure and two subplots

axes[0].imshow(x[0].permute(1,2,0))
axes[1].imshow(x[1].permute(1,2,0))

plt.show()


NameError: name 'data' is not defined